# Notebook 6 — Klarna: When Does the AI Actually Save Money?
**AI Models and Agents for Business · Week 3 · Session 6**

In February 2024 Klarna said its AI assistant handled **2.3 million conversations** in its first month, **two-thirds** of its customer service chats, doing the work of **700 full-time agents**, with **25% fewer repeat inquiries**. These are the company's own claims. In May 2025 its CEO said Klarna would hire human support staff again, so customers could always reach a person.

Both can be true at once, because **cheap per chat is not the same as cheap per resolved issue.** Today you build the cost model that shows why.

**No API key needed.** This notebook is pure business arithmetic.

### What you will do
1. See how a support system decides which chats go to a person (Part A).
2. Build the cost per issue with AI, and compare it with a person handling the issue from the start (Part B).
3. Find the **break-even**: the escalation rate, or quality level, at which the AI stops saving money (Part C).
4. Add the cost a ticket never shows: customers who leave after a bad experience (Part D).
5. Write one sentence on what Klarna got wrong, or right (Part E).

### Ground rules
- Run cells top to bottom. Cells marked **✏️ YOUR TURN** are yours to edit.
- Every cost below is an **illustrative assumption**, not a Klarna figure. Changing them is the point.

*Sources: Klarna press release, Feb 27, 2024; Siemiatkowski interview with Bloomberg, as reported by Entrepreneur, May 9, 2025.*

---
## Part A — Which chats go to a person?

A real deployment routes each chat. Below is a simple three-tier router: the AI answers alone, the AI drafts and a person approves, or a person takes over. The confidence scores are made up for illustration.

In [ ]:
# Cell 1 — A three-tier router. Run this first.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

tickets = [
    {"id": 101, "text": "What is my current balance and next due date?", "sentiment_risk": 0.10, "dispute_amount": 0},
    {"id": 102, "text": "I was charged twice for my purchase and I am furious! Fix this now.", "sentiment_risk": 0.95, "dispute_amount": 450},
    {"id": 103, "text": "How do I update my shipping address for my pending order?", "sentiment_risk": 0.20, "dispute_amount": 0},
    {"id": 104, "text": "My item arrived broken and I want a full refund immediately.", "sentiment_risk": 0.75, "dispute_amount": 120},
    {"id": 105, "text": "Can I extend my payment plan by 2 weeks?", "sentiment_risk": 0.40, "dispute_amount": 50},
]
confidence = [0.92, 0.88, 0.91, 0.72, 0.81]   # how sure the AI is of its answer (illustrative)

def route(ticket, conf):
    """Tier 3 if the AI is unsure, the customer is upset, or a lot of money is at stake."""
    if conf < 0.60 or ticket["sentiment_risk"] >= 0.80 or ticket["dispute_amount"] > 300:
        return "3: person takes over"
    if conf < 0.85 or 100 <= ticket["dispute_amount"] <= 300:
        return "2: AI drafts, person approves"
    return "1: AI answers alone"

routed = pd.DataFrame([{"Ticket": t["id"], "Customer wrote": t["text"][:48] + "...",
                        "AI confidence": c, "Upset (0-1)": t["sentiment_risk"],
                        "Dispute ($)": t["dispute_amount"], "Tier": route(t, c)}
                       for t, c in zip(tickets, confidence)])
display(routed)
print(f"Handled by the AI alone: {(routed.Tier.str.startswith('1')).mean():.0%} of these tickets")

💬 **Discuss:** Every ticket that reaches Tier 2 or 3 costs you the AI **and** a person. What would happen to the share at Tier 1 if you lowered the confidence bar from 0.85 to 0.70? What could go wrong?

---
## Part B — The cost of one issue

An AI chat is cheap, but not every chat ends there.

**Cost per issue with AI = AI cost + (escalation rate × human cost) + (AI repeat rate × cost to handle it again)**

**Cost per issue with a person = human cost + (person repeat rate × cost to handle it again)**

**Follow one issue through each process:**
- **AI first:** every issue starts with the AI, so you pay the AI cost. A share is **escalated** to a person, who handles it start to finish: you pay for the AI *and* the person. Separately, a share of issues **come back** after being closed because the first answer was poor; a person handles the comeback, and you pay again.
- **Person first:** a person handles the issue from the start. People get things wrong too, so some of these issues also come back, and you pay again.

The AI only loses on quality if its repeat rate is **higher** than a person's. That is why both rates are in the model.

**Unit of analysis:** everything here is per **issue** (one customer problem), not per chat. One issue can take several chats, so a count of chats overstates the number of distinct issues.

In [ ]:
# Cell 2 — ✏️ YOUR TURN: the assumptions (all illustrative; change them)
AI_COST_PER_CHAT = 0.05   # model and hosting cost per conversation ($)
HUMAN_COST       = 7.00   # a person handles one issue from start to finish ($)
ESCALATION_RATE  = 0.33   # share of AI chats handed to a person (Klarna said the AI handled about two-thirds)
REPEAT_RATE      = 0.15   # share of AI-first issues that come back because the first answer was poor
HUMAN_REPEAT_RATE = 0.05  # share of person-first issues that come back (people make mistakes too)
REDO_COST        = 7.00   # cost to handle a repeat, usually by a person ($)

ISSUES_PER_MONTH = 2_300_000   # scale only: Klarna reported 2.3M AI *conversations* in month one,
                               # and since some issues take several chats, distinct issues were fewer

def cost_with_ai(ai=None, human=None, esc=None, rep=None, redo=None):
    ai    = AI_COST_PER_CHAT if ai is None else ai
    human = HUMAN_COST if human is None else human
    esc   = ESCALATION_RATE if esc is None else esc
    rep   = REPEAT_RATE if rep is None else rep
    redo  = REDO_COST if redo is None else redo
    return ai + esc * human + rep * redo

def cost_person(rep_h=None, redo=None):
    rep_h = HUMAN_REPEAT_RATE if rep_h is None else rep_h
    redo  = REDO_COST if redo is None else redo
    return HUMAN_COST + rep_h * redo

with_ai = cost_with_ai()
person = cost_person()
print(f"Cost per issue, AI first : ${with_ai:,.2f}")
print(f"   AI                    : ${AI_COST_PER_CHAT:,.2f}")
print(f"   escalations           : ${ESCALATION_RATE * HUMAN_COST:,.2f}")
print(f"   repeats               : ${REPEAT_RATE * REDO_COST:,.2f}")
print(f"Cost per issue, person   : ${person:,.2f}")
print(f"   person                : ${HUMAN_COST:,.2f}")
print(f"   repeats               : ${HUMAN_REPEAT_RATE * REDO_COST:,.2f}")
print()
saving = person - with_ai
print(f"Saving per issue: ${saving:,.2f} ({saving / person:.0%})")
print(f"At {ISSUES_PER_MONTH:,} issues a month: ${saving * ISSUES_PER_MONTH:,.0f} a month")

💬 **Discuss:** Which of the three AI terms is biggest? Is it the AI's price per chat?

---
## Part C — Find the break-even

The break-even is the point where the AI-first cost equals the person-first cost. Hold everything else fixed and solve for one input at a time.

In [ ]:
# Cell 3 — Break-even escalation rate and break-even repeat rate
person = cost_person()
be_escalation = (person - AI_COST_PER_CHAT - REPEAT_RATE * REDO_COST) / HUMAN_COST
be_repeat     = (person - AI_COST_PER_CHAT - ESCALATION_RATE * HUMAN_COST) / REDO_COST

print(f"With repeats at {REPEAT_RATE:.0%}, the AI stops saving money when escalations reach {be_escalation:.0%}.")
print(f"With escalations at {ESCALATION_RATE:.0%}, the AI stops saving money when its repeat rate reaches {be_repeat:.0%}"
      f" (a person's is {HUMAN_REPEAT_RATE:.0%}).")

rates = np.linspace(0, 1, 101)
fig, ax = plt.subplots(figsize=(8, 4))
for esc in [0.20, ESCALATION_RATE, 0.50]:
    ax.plot(rates, [cost_with_ai(esc=esc, rep=r) for r in rates], label=f"AI first, {esc:.0%} escalated")
ax.axhline(person, color="black", linestyle="--", label="Person first")
ax.set_xlabel("AI repeat rate (share of AI-first issues that come back)")
ax.set_ylabel("Cost per issue ($)")
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_title("Where the AI stops saving money")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Cell 4 — Which input moves the answer most?
# Each input is moved from half its value to one and a half times its value; everything else stays fixed.
inputs = {"AI cost per chat": "ai", "Escalation rate": "esc", "Repeat rate": "rep", "Cost to redo": "redo"}
base = {"ai": AI_COST_PER_CHAT, "esc": ESCALATION_RATE, "rep": REPEAT_RATE, "redo": REDO_COST}
rows = []
for label, key in inputs.items():
    low = cost_with_ai(**{key: base[key] * 0.5})
    high = cost_with_ai(**{key: base[key] * 1.5})
    rows.append({"Input": label, "Low (x0.5)": low, "High (x1.5)": high, "Swing ($)": high - low})
swing = pd.DataFrame(rows).set_index("Input").sort_values("Swing ($)", ascending=False)
display(swing.style.format("${:,.2f}"))

### ✅ Check before you move on
Name the input that moves the cost per issue most. Is it the AI's price, or something about the *quality* of the AI's work (how often it escalates, how often issues come back)?

---
## Part D — The cost a ticket never shows

Ticket costs leave out the customer. A customer whose problem comes back is more likely to leave, and a lost customer costs far more than a support ticket. Add that to the cost of a repeat.

In [ ]:
# Cell 5 — ✏️ YOUR TURN: what is a bad experience worth?
LEAVE_RATE_AFTER_REPEAT = 0.10   # share of customers with a repeat problem who stop using the service
CUSTOMER_VALUE          = 300    # future profit from a customer who stays ($)

true_redo = REDO_COST + LEAVE_RATE_AFTER_REPEAT * CUSTOMER_VALUE
with_ai_full = cost_with_ai(redo=true_redo)
person_full = cost_person(redo=true_redo)      # person-first repeats lose customers too
be_repeat_full = (person_full - AI_COST_PER_CHAT - ESCALATION_RATE * HUMAN_COST) / true_redo

print(f"True cost of a repeat: ${true_redo:,.2f} (ticket ${REDO_COST:,.2f} + lost customers ${LEAVE_RATE_AFTER_REPEAT * CUSTOMER_VALUE:,.2f})")
print(f"Cost per issue, counting lost customers: AI first ${with_ai_full:,.2f}  vs. person first ${person_full:,.2f}")
print(f"Break-even AI repeat rate: {be_repeat:.0%} on ticket cost alone, {be_repeat_full:.0%} counting lost customers")
print(f"(A person's repeat rate is {HUMAN_REPEAT_RATE:.0%}.)")

💬 **Discuss:** How far did the break-even move? Why might a company see the saving in its support budget within a month, but see the lost customers only a year later, in a different team's numbers?

---
## Part E — ✏️ YOUR TURN: what did Klarna get wrong, or right?

Klarna said the AI handled about two-thirds of chats (escalation about one-third) and that repeat inquiries fell 25%. In this model, that means an AI repeat rate 25% *below* a person's: `REPEAT_RATE = 0.75 * HUMAN_REPEAT_RATE`. Try those claims in your model, then try a world where quality was worse than claimed. What would have to be true for the reversal in 2025 to make sense?

Write **one sentence** below that uses a number from your model.

> *My sentence:* ...

---
## ✅ Notebook 6 complete — Reflection (submit with your notebook)

These prep you for **Quiz 2** (Session 8).

1. With your assumptions, what were the break-even escalation rate and the break-even repeat rate?
2. Which input moved the cost per issue most, and what does that say about where to spend effort in an AI support deployment?
3. Explain in two sentences why "cheap per chat" and "cheap per resolved issue" can give opposite answers.
4. Your one sentence on Klarna, from Part E.

*Next session: The Ladder of AI Use, and Project 1 launches.*